### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the
web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON
schema)
2. A function or coroutine to execute.

In [4]:
import os
from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.6-27b")
resonse=model.invoke("Why do parrots talks?")
resonse

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talks?" (Note: grammatical error "talks" should be "talk", but the meaning is clear). They want to know the reasons behind parrots\' ability and tendency to mimic human speech.\n\n2.  **Identify Key Concepts**:\n   - Parrots\' vocal anatomy (syrinx)\n   - Cognitive abilities (intelligence, social learning)\n   - Evolutionary/biological reasons (communication in the wild)\n   - Social bonding/motivation\n   - Captivity vs. wild behavior\n   - Scientific research findings\n\n3.  **Structure the Response**:\n   - Acknowledge the question\n   - Explain the biological/physical capability\n   - Explain the cognitive/social reasons\n   - Differentiate wild vs. captive behavior\n   - Mention scientific insights\n   - Keep it clear, accurate, and engaging\n\n4.  **Draft - Section by Section**:\n   *(Physical Capability)* Parrots have a specialized vocal organ called the s

In [7]:
from langchain.tools import tool


@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""

    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])    

In [13]:
response=model_with_tools.invoke("What's the weather like in Delhi?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in Delhi.\n2.  Check available tools: `get_weather` function is available.\n3.  Extract parameters: `location` = "Delhi".\n4.  Call the function: `get_weather(location="Delhi")`.\n5.  Formulate response based on the function\'s output. (Wait for function output first) -> I will generate the tool call.✅\n', 'tool_calls': [{'id': '00sw1r3zm', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 276, 'total_tokens': 407, 'completion_time': 0.256729496, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.022378231, 'prompt_tokens_details': None, 'queue_time': 0.047434298, 'total_time': 0.279107727}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_60e0d2f248', 'service_tier': 'on_demand', 'finish_reason

### Tool Execution Loops

In [14]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Delhi?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)
# "The current weather in Boston is 72°F and sunny."

It's currently sunny in Delhi.


In [15]:
messages

[{'role': 'user', 'content': "What's the weather in Delhi?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user is asking for the weather in Delhi.\n2.  Check available tools: I have a `get_weather` tool that takes a `location` parameter.\n3.  Extract parameters: Location = "Delhi".\n4.  Call the tool: `get_weather(location="Delhi")`.\n5.  Formulate response based on the tool\'s output. (I will execute the tool call now)✅\n', 'tool_calls': [{'id': '938ymhr9x', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 275, 'total_tokens': 406, 'completion_time': 0.248548851, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.019547527, 'prompt_tokens_details': None, 'queue_time': 0.046973293, 'total_time': 0.268096378}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': '